In [1]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn.cluster as cluster
import umap
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import PPBuilder
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import pearsonr

/home/kunzj/venv/test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def get_whole_seq(binder_idx,iteration='iteration_0'):
    # Load the PDB file
    parser = PDBParser(QUIET=True)
    ppb = PPBuilder()
    pattern = f"/home/kunzj/internship_uva_compchem/Data/{iteration}/bindcraft/Accepted/Ranked/{str(binder_idx)}_BAX_1F16_*.pdb"
    file = glob.glob(pattern)
    structure = parser.get_structure("protein", file[0])
    chains = ppb.build_peptides(structure)
    seq_prot = chains[0].get_sequence()
    seq_pept = chains[1].get_sequence()
    return seq_prot, seq_pept

### data retrieval

In [ ]:
df = pd.read_csv("/home/kunzj/internship_uva_compchem/Data/iteration_0/score/md_data_all_binders_raw.csv", index_col=0)

df.set_index("Binder_ID", inplace=True)
features = ["energies", "rmsd", "bsa", "hbond", "sbridge"]
n_clusters = 4
df["seq_prot"] = str
df["seq_pept"] = str
for idx in df.index:
    seq_prot, seq_pept = get_whole_seq(idx)
    df.loc[idx, "seq_prot"] = str(seq_prot)
    df.loc[idx, "seq_pept"] = str(seq_pept)



# features which are already +1 as best value
for col in ["sbridge", "hbond"]:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    df[col] = scaler.fit_transform(df[[col]])

# features which need to be inverted so that best value is at +1
for col in ["energies", "bsa", "rmsd"]:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    df[col] = -scaler.fit_transform(df[[col]])

### own score creation

In [21]:
print("Pearson correlation coefficients:")
print("----------------------------------")
for val in ["hbond", "rmsd", "sbridge", "bsa"]:
    r, p = pearsonr(list(df["energies"]), list(df[val]))
    print(f"{val} ~ r={r:.3f}, p={p:.2e}")
    print("")
print("----------------------------------")
weights = {
    "energies": 2,
    "hbond": 1.59,
    "rmsd": 1.347,
    "sbridge": 1.166,
    "bsa": 0.9,
}
print("Chosen Weights for final score calculation:")
print("----------------------------------")
for k, v in weights.items():
    print(f"{k}: {v}")
    print("")
print("----------------------------------")


Pearson correlation coefficients:
----------------------------------
hbond ~ r=0.590, p=6.51e-11

rmsd ~ r=0.347, p=3.60e-04

sbridge ~ r=0.166, p=9.58e-02

bsa ~ r=-0.100, p=3.15e-01

----------------------------------
Chosen Weights for final score calculation:
----------------------------------
energies: 2

hbond: 1.59

rmsd: 1.347

sbridge: 1.166

bsa: 0.9

----------------------------------


In [ ]:
df["final_score"] = (
    df["energies"] * weights["energies"]
    + df["rmsd"] * weights["rmsd"]
    + df["bsa"] * weights["bsa"]
    + df["hbond"] * weights["hbond"]
    + df["sbridge"] * weights["sbridge"]
)
scaler = MinMaxScaler(feature_range=(-1, 1))
df["final_score"] = scaler.fit_transform(df[["final_score"]])
df[["final_score", "seq_prot", "seq_pept"]].to_csv(
    "/home/kunzj/internship_uva_compchem/Data/iteration_0/finished_data_cosine_sim.csv"
)

### dataset creation for second iteration

In [ ]:
df_first_iteration = pd.read_csv("/home/kunzj/internship_uva_compchem/Data/iteration_0/score/md_data_all_binders_raw.csv", index_col=0)

df_first_iteration.set_index("Binder_ID", inplace=True)
features = ["energies", "rmsd", "bsa", "hbond", "sbridge"]

df_first_iteration["seq_prot"] = str
df_first_iteration["seq_pept"] = str
for idx in df_first_iteration.index:
    seq_prot, seq_pept = get_whole_seq(idx)
    df_first_iteration.loc[idx, "seq_prot"] = str(seq_prot)
    df_first_iteration.loc[idx, "seq_pept"] = str(seq_pept)


df_second_iteration = pd.read_csv("/home/kunzj/internship_uva_compchem/Data/iteration_1/score/md_data_all_binders_raw.csv", index_col=0)

df_second_iteration.set_index("Binder_ID", inplace=True)
features = ["energies", "rmsd", "bsa", "hbond", "sbridge"]

df_second_iteration["seq_prot"] = str
df_second_iteration["seq_pept"] = str
for idx in df_second_iteration.index:
    seq_prot, seq_pept = get_whole_seq(idx,iteration='iteration_1')
    df_second_iteration.loc[idx, "seq_prot"] = str(seq_prot)
    df_second_iteration.loc[idx, "seq_pept"] = str(seq_pept)
    

df = pd.concat([df_first_iteration, df_second_iteration])

### normal procedure ###

# features which are already +1 as best value
for col in ["sbridge", "hbond"]:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    df[col] = scaler.fit_transform(df[[col]])

# features which need to be inverted so that best value is at +1
for col in ["energies", "bsa", "rmsd"]:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    df[col] = -scaler.fit_transform(df[[col]])
    
print("Pearson correlation coefficients:")
print("----------------------------------")
for val in ["hbond", "rmsd", "sbridge", "bsa"]:
    r, p = pearsonr(list(df["energies"]), list(df[val]))
    print(f"{val} ~ r={r:.3f}, p={p:.2e}")
    print("")
print("----------------------------------")


Pearson correlation coefficients:
----------------------------------
hbond ~ r=0.552, p=1.22e-17

rmsd ~ r=0.275, p=6.76e-05

sbridge ~ r=0.117, p=9.53e-02

bsa ~ r=-0.064, p=3.65e-01

----------------------------------


In [ ]:
weights = {
    "energies": 2,
    "hbond": 1.55,
    "rmsd": 1.28,
    "sbridge": 1.12,
    "bsa": 0.94,
}
print("Chosen Weights for final score calculation:")
print("----------------------------------")
for k, v in weights.items():
    print(f"{k}: {v}")
    print("")
print("----------------------------------")


df["final_score"] = (
    df["energies"] * weights["energies"]
    + df["rmsd"] * weights["rmsd"]
    + df["bsa"] * weights["bsa"]
    + df["hbond"] * weights["hbond"]
    + df["sbridge"] * weights["sbridge"]
)
scaler = MinMaxScaler(feature_range=(-1, 1))
df["final_score"] = scaler.fit_transform(df[["final_score"]])
df[["final_score", "seq_prot", "seq_pept"]].to_csv(
    "./data/iteration_1/finished_data_cosine_sim.csv"
)

Chosen Weights for final score calculation:
----------------------------------
energies: 2

hbond: 1.55

rmsd: 1.28

sbridge: 1.12

bsa: 0.94

----------------------------------
